In [1]:
import os
import yaml
import copy
import time
import numpy as np
import pandas as pd
import xarray as xr

In [2]:
dict_rename = {
    'PRECT_30d_default': 'PRECT_30d',
    'PRECT_max_default': 'PRECT_max',
    'PRECT_mean_default': 'PRECT_mean',
    'TREFHTMN_min_default': 'T2_1h_min',
    'TREFHTMX_max_default': 'T2_1h_max',
    'TREFHT_30d_default': 'T2_30d_max',
    'TREFHT_max_default': 'T2_1d_max',
    'TREFHT_mean_default': 'T2_1d_mean',
    'TREFHT_min_default': 'T2_1d_min'
}

In [32]:
# ============================================================== #
# Save all
# ============================================================== #
stn_names = ['Fairbanks', 'Pituffik', 'Guam', 'Yuma_PG', 'Fort_Bragg']

list_ds = []
for stn in stn_names:
    fn1 = f'/glade/derecho/scratch/ksha/EPRI_data/METRICS/{stn}/' + "CESM_STN.zarr"
    ds1 = xr.open_zarr(fn1)
    ds1 = ds1.rename({v: f'{stn}_{v}' for v in ds1.data_vars})
    
    try:
        ds1 = ds1.drop_vars(['lat', 'lon'], errors='ignore')
    except:
        pass
        
    if "valid_year" in ds1.data_vars:
        ds1 = ds1.set_coords("valid_year")
    
    fn2 = f'/glade/derecho/scratch/ksha/EPRI_data/METRICS_STN/{stn}/' + "CESM_member_metrics.zarr"
    ds2 = xr.open_zarr(fn2)
    ds2 = ds2.rename(dict_rename)
    ds2 = ds2.rename({'init_time': 'init_year'})
    ds2 = ds2.rename({v: f'{stn}_simple_{v}' for v in ds2.data_vars})
    
    try:
        ds2 = ds2.drop_vars(['lat', 'lon'], errors='ignore')
    except:
        pass
        
    if "valid_year" in ds1.data_vars:
        ds2 = ds2.set_coords("valid_year")

    ds = xr.merge([ds1, ds2])
    ds = ds.transpose('init_year', 'lead_year', 'member')
    
    list_ds.append(ds)

ds_all = xr.merge(list_ds)

ds_all = ds_all.drop_vars(['quantile',])
ds_all = ds_all.chunk({'init_year': -1, 'lead_year': -1, 'member': -1})

save_name = '/glade/derecho/scratch/ksha/EPRI_data/METRICS/STN_CESM_ALL_20260604.zarr'
ds_all.to_zarr(save_name, mode='w')
print(save_name)

/glade/derecho/scratch/ksha/EPRI_data/METRICS/STN_CESM_ALL_20260604.zarr


In [33]:
ds_all

<xarray.Dataset> Size: 4MB
Dimensions:                          (init_year: 62, lead_year: 10, member: 20)
Coordinates:
  * init_year                        (init_year) int64 496B 1959 1960 ... 2020
  * lead_year                        (lead_year) int64 80B 0 1 2 3 4 5 6 7 8 9
  * member                           (member) int64 160B 11 12 13 ... 28 29 30
Data variables: (12/98)
    Fairbanks_T2_1d_max              (init_year, lead_year, member) float32 50kB dask.array<chunksize=(62, 10, 20), meta=np.ndarray>
    Fairbanks_T2_1d_max_ensmean      (init_year, lead_year) float32 2kB dask.array<chunksize=(62, 10), meta=np.ndarray>
    Fairbanks_T2_1h_max              (init_year, lead_year, member) float32 50kB dask.array<chunksize=(62, 10, 20), meta=np.ndarray>
    Fairbanks_T2_1h_max_ensmean      (init_year, lead_year) float32 2kB dask.array<chunksize=(62, 10), meta=np.ndarray>
    Fairbanks_T2_30d_max             (init_year, lead_year, member) float32 50kB dask.array<chunksize=(62, 10, 20), meta=np.ndarray>
    Fairbanks_T2_30d_max_ensmean     (init_year, lead_year) float32 2kB dask.array<chunksize=(62, 10), meta=np.ndarray>
    ...                               ...
    Fort_Bragg_simple_T2_1h_min      (init_year, lead_year, member) float32 50kB dask.array<chunksize=(62, 10, 20), meta=np.ndarray>
    Fort_Bragg_simple_T2_1h_max      (init_year, lead_year, member) float32 50kB dask.array<chunksize=(62, 10, 20), meta=np.ndarray>
    Fort_Bragg_simple_T2_30d_max     (init_year, lead_year, member) float32 50kB dask.array<chunksize=(62, 10, 20), meta=np.ndarray>
    Fort_Bragg_simple_T2_1d_max      (init_year, lead_year, member) float32 50kB dask.array<chunksize=(62, 10, 20), meta=np.ndarray>
    Fort_Bragg_simple_T2_1d_mean     (init_year, lead_year, member) float32 50kB dask.array<chunksize=(62, 10, 20), meta=np.ndarray>
    Fort_Bragg_simple_T2_1d_min      (init_year, lead_year, member) float32 50kB dask.array<chunksize=(62, 10, 20), meta=np.ndarray>
Attributes:
    cell_methods:  time: maximum
    long_name:     Maximum reference height temperature over output period
    units:         K

## ERA5

In [23]:
stn_names = ['Fairbanks', 'Pituffik', 'Guam', 'Yuma_PG', 'Fort_Bragg']

list_ds = []
for stn in stn_names:
    fn1 = f'/glade/derecho/scratch/ksha/EPRI_data/METRICS/{stn}/' + "ERA5_STN.zarr"
    ds1 = xr.open_zarr(fn1)
    ds1 = ds1.rename({v: f'{stn}_{v}' for v in ds1.data_vars})
    
    try:
        ds1 = ds1.drop_vars(['lat', 'lon'], errors='ignore')
        ds1 = ds1.drop_vars(['latitude', 'longitude'], errors='ignore')
    except:
        pass
        
    # keep valid_year shared across stations (don't prefix it)
    if "valid_year" in ds1.data_vars:
        ds1 = ds1.set_coords("valid_year")

    fn2 = f'/glade/derecho/scratch/ksha/EPRI_data/METRICS_STN/{stn}/' + "ERA5_metrics.zarr"
    ds2 = xr.open_zarr(fn2)
    ds2 = ds2.rename(dict_rename)
    ds2 = ds2.rename({'year': 'valid_year'})
    ds2 = ds2.rename({v: f'{stn}_simple_{v}' for v in ds2.data_vars})
    
    try:
        ds2 = ds2.drop_vars(['lat', 'lon'], errors='ignore')
        ds2 = ds2.drop_vars(['latitude', 'longitude'], errors='ignore')
    except:
        pass
        
    # keep valid_year shared across stations (don't prefix it)
    if "valid_year" in ds2.data_vars:
        ds2 = ds2.set_coords("valid_year")
    
    ds = xr.merge([ds1, ds2])
    
    list_ds.append(ds)

ds_all = xr.merge(list_ds)

ds_all = ds_all.drop_vars(['quantile',])
ds_all = ds_all.chunk({'valid_year': -1})
save_name = '/glade/derecho/scratch/ksha/EPRI_data/METRICS/STN_ERA5_ALL_20260604.zarr'
ds_all.to_zarr(save_name, mode='w')
print(save_name)

/glade/derecho/scratch/ksha/EPRI_data/METRICS/STN_ERA5_ALL_20260604.zarr
